# 06 GPU Acceleration with CuPy

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/high_performance_python/04_GPU_Acceleration_with_CuPy.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=high_performance_python/04_GPU_Acceleration_with_CuPy.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)


In [ ]:
%config InlineBackend.figure_format = "retina"
# === Environment Setup ===
import timeit

import numpy as np

# --- Configuration ---
# No specific configuration needed for CuPy beyond availability check


## The Lens: Massive Parallelism on GPUs
**What economic problem are we solving?**
Some computational tasks in economics—like solving high-dimensional dynamic programming problems, training deep neural networks, or inverting massive matrices for econometrics—are simply too slow on a standard CPU. A CPU is designed to do a few complex things very quickly (low latency), but these tasks require doing *billions* of simple things simultaneously (high throughput).

**Why do we need this method?**
**Graphics Processing Units (GPUs)** are architected for exactly this kind of massive parallelism. With thousands of cores, a GPU can perform matrix operations materially faster on sufficiently large, suitable workloads than a CPU. **CuPy** brings this power to Python with an API that mirrors NumPy. This can accelerate sufficiently large, GPU-suitable kernels, but performance is workload- and hardware-dependent. Benchmark end-to-end runtime, including host-device transfers, before drawing conclusions.

**Economic question.** In *06 GPU Acceleration with CuPy*, what must remain economically invariant when the computational representation changes? Performance matters when computation changes which economic questions are feasible to ask. The relevant objective is not a synthetic speedup alone but lower time-to-solution at fixed numerical accuracy and reproducibility. Benchmark with representative problem sizes, separate compilation or transfer overhead from steady-state work, and verify that the optimized implementation agrees with a clear reference calculation.

### Learning Objectives
- **Measure** the performance bottleneck in 06 GPU Acceleration with CuPy before optimizing it.
- **Implement** a faster version while preserving a simple reference implementation.
- **Separate** setup/compilation/transfer overhead from steady-state execution time.
- **Validate** numerical equivalence and explain when the optimization is economically worthwhile.

### Prerequisites
- **`../01-Foundations/12_NumPy.ipynb`**: vectorized arrays, broadcasting, and memory layout.
- **`../01-Foundations/23_Profiling_and_Performance.ipynb`**: profiling and benchmark discipline.
- Familiarity with functions, NumPy, and reproducible timing experiments.
* **Learning-path prerequisite:** [`03_Parallel_Computing_with_Dask.ipynb`](03_Parallel_Computing_with_Dask.ipynb)


> **Learning path:** Building on [`03_Parallel_Computing_with_Dask.ipynb`](03_Parallel_Computing_with_Dask.ipynb); this notebook closes the current track.


### Table of Contents
1.  [CPU vs. GPU: A Tale of Two Processors](#1.-CPU-vs.-GPU:-A-Tale-of-Two-Processors)
2.  [The CuPy API: NumPy on the GPU](#2.-The-CuPy-API:-NumPy-on-the-GPU)
3.  [Moving Data Between CPU and GPU](#3.-Moving-Data-Between-CPU-and-GPU)
4.  [Benchmarking: CPU vs. GPU Performance](#4.-Benchmarking:-CPU-vs.-GPU-Performance)
5.  [A More Complex Example: SVD](#5.-A-More-Complex-Example:-SVD)
6.  [Summary](#6.-Summary)


### 1. CPU vs. GPU: A Tale of Two Processors

Central Processing Units (CPUs) and Graphics Processing Units (GPUs) are designed with fundamentally different architectures for different purposes.

- **CPUs** are composed of a few, very powerful cores optimized for **sequential, latency-sensitive tasks**. They are masters of executing complex instructions one after another very quickly.
- **GPUs**, on the other hand, are composed of thousands of smaller, simpler cores designed for **parallel, throughput-sensitive tasks**. They excel at performing the same simple operation on thousands of data points simultaneously.

Many operations in scientific computing and economic modeling, such as matrix multiplication, vector operations, and simulations, are inherently parallel. By offloading these tasks to a GPU, we can often achieve massive performance gains over a CPU.

**CuPy** is a Python library that makes GPU computing incredibly accessible. It provides a **drop-in replacement for NumPy**, meaning its API is designed to be a near-perfect mirror of NumPy's. This allows you to accelerate your existing NumPy code on an NVIDIA GPU with minimal code changes.

--- 
**<font color='red'>IMPORTANT NOTE:</font>** To run this notebook, you **must** have a compatible **NVIDIA GPU** and the correct version of the **NVIDIA CUDA Toolkit** installed on your system. If you do not have the required hardware and software, the code cells in this notebook will fail. You can still read the content to understand the concepts, but you will not be able to execute the code.


### 2. The CuPy API: NumPy on the GPU

The convention is to import NumPy as `np` and CuPy as `cp`. You can then create arrays on the GPU using familiar syntax.


In [ ]:
try:
    import cupy as cp
except ImportError:
    cp = None
    CUPY_AVAILABLE = False
else:
    try:
        gpu_name = cp.cuda.runtime.getDeviceProperties(0)["name"].decode("utf-8")
        CUPY_AVAILABLE = True
        print(f"> **Note:** Found compatible GPU: {gpu_name}")
    except cp.cuda.runtime.CUDARuntimeError:
        CUPY_AVAILABLE = False
        print("> **Note:** CuPy is installed, but no compatible NVIDIA GPU is available.")

if CUPY_AVAILABLE:
    # Create a NumPy array on the CPU
    x_cpu = np.arange(10)
    print(f"NumPy array on CPU: {x_cpu}")

    # Create a CuPy array on the GPU
    x_gpu = cp.arange(10)
    print(f"CuPy array on GPU: {x_gpu}")


### 3. Moving Data Between CPU and GPU

A critical concept in GPU computing is data transfer. For the GPU to operate on data, that data must first be moved from the host (CPU) memory to the device (GPU) memory. This transfer has a cost, so efficient GPU computing often involves minimizing CPU-GPU data transfers.

- To move a NumPy array to the GPU, use `cupy.asarray()`.
- To move a CuPy array back to the CPU, use `cupy.asnumpy()` or the `.get()` method.


In [ ]:
if CUPY_AVAILABLE:
    # Create a NumPy array
    numpy_arr = np.random.rand(5)
    print(f"Original NumPy array: {numpy_arr}")

    # Move it to the GPU
    cupy_arr = cp.asarray(numpy_arr)
    print(f"CuPy array on GPU: {cupy_arr}")

    # Move it back to the CPU
    numpy_arr_back = cp.asnumpy(cupy_arr)
    print(f"Array back on CPU: {numpy_arr_back}")


### 4. Benchmarking: CPU vs. GPU Performance

Let's demonstrate the performance difference with a classic example: multiplying two large matrices. This is a highly parallelizable task where GPUs excel.


In [ ]:
if CUPY_AVAILABLE:
    # Define the size of the matrices
    size = 5000

    # Create two random matrices in NumPy (CPU)
    a_cpu = np.random.rand(size, size).astype(np.float32)
    b_cpu = np.random.rand(size, size).astype(np.float32)

    # Create two random matrices in CuPy (GPU)
    a_gpu = cp.random.rand(size, size).astype(cp.float32)
    b_gpu = cp.random.rand(size, size).astype(cp.float32)

    print("Benchmarking Matrix Multiplication...")


Now, let's time the matrix multiplication on the CPU.


In [ ]:
if CUPY_AVAILABLE:
    cpu_time = timeit.timeit(lambda: np.dot(a_cpu, b_cpu), number=10) # Matrix multiplication
    print(f"CPU time: {cpu_time:.4f} seconds")


And now, let's time the same operation on the GPU. Note that for a fair comparison, we should ensure the data is already on the GPU. We also need to synchronize the device to ensure the computation is finished before stopping the timer.


In [ ]:
if CUPY_AVAILABLE:
    cp.cuda.runtime.deviceSynchronize()
    gpu_time = timeit.timeit(lambda: cp.dot(a_gpu, b_gpu), number=10)
    cp.cuda.runtime.deviceSynchronize()
    print(f"GPU time: {gpu_time:.4f} seconds")
    print(f"> **Note:** Speedup: {cpu_time / gpu_time:.1f}x")


You should observe a dramatic speedup, potentially 50-a workload-dependent factor or more, depending on your specific CPU and GPU. This is the power of offloading highly parallelizable work to the GPU.


### 5. A More Complex Example: SVD

The benefits extend to more complex linear algebra, which is at the heart of many econometric and statistical methods.


In [ ]:
if CUPY_AVAILABLE:
    print("Benchmarking SVD...")
    cpu_svd_time = timeit.timeit(lambda: np.linalg.svd(a_cpu), number=1)
    print(f"CPU SVD time: {cpu_svd_time:.4f} seconds")

    cp.cuda.runtime.deviceSynchronize()
    gpu_svd_time = timeit.timeit(lambda: cp.linalg.svd(a_gpu), number=1)
    cp.cuda.runtime.deviceSynchronize()
    print(f"GPU SVD time: {gpu_svd_time:.4f} seconds")
    print(f"> **Note:** SVD Speedup: {cpu_svd_time / gpu_svd_time:.1f}x")


## Exercises

**1. Mechanism and assumptions (Conceptual):** Build a baseline for **06 GPU Acceleration with CuPy** and predict its time and memory complexity before benchmarking. State which measurement would falsify your performance hypothesis.

**2. Reproduce and diagnose (Applied):** Optimize the workload using 1. CPU vs. GPU: A Tale of Two Processors, 2. The CuPy API: NumPy on the GPU. Report warm-up separately from steady-state timing, use multiple repetitions, and verify numerical equivalence to the baseline.

**3. Robust extension (Challenge):** Scale the workload until the bottleneck changes (compute, memory bandwidth, serialization, transfer, or scheduler overhead). Identify the crossover point and recommend when the optimization should not be used.

<details>
<summary>Solution guidance</summary>

A strong solution states assumptions before computation, includes an independent diagnostic or limiting-case check, and interprets the result in the units of the economic problem. For the challenge, separate changes caused by the economic assumption from changes caused by numerical approximation or tuning.

</details>


---
## Summary

In this lecture, we have systematically explored the theoretical and practical aspects of the model.

**Key Takeaways:**
1.  **Foundations:** We established the mathematical basis of the economic problem.
2.  **Computation:** We implemented the solution using efficient algorithms.
3.  **Implications:** We analyzed the economic significance of the results.

**Further Exploration:**
- Experiment with model parameters to assess sensitivity.
- Extend the framework by relaxing simplifying assumptions.


## References & Further Reading

- Gorelick, M. & Ozsvald, I. (2020). *High Performance Python* (2nd ed.). O'Reilly.
- Numba project. *Numba Documentation*.
- Dask Development Team. *Dask Documentation*; CuPy Developers. *CuPy Documentation*.
